## 5.1 Machine Learning & Modeling: The Big Picture

**Definitions**.

<u>Model</u>: A mathematical equation that takes data values as inputs and predicts a data value as output.

<u>Training/Learning</u>: Models have *parameters* that are *learned* by analyzing existing data. The accuracy of these parameter values (along with having a good functional form, ie, choosing the appropriate model specification for the problem at hand) are what ultimately makes a model successful or not.

<u>Algorithm</u>: A process or set of rules to be followed in calculations or other problem-solving operations, especially by a computer.

<u>Supervised Learning</u>: Algorithm builds a model by  training on *labeled* data, then its accuracy/effectiveness is tested by having the algorithm predict labels on other data and measuring how well it does.

<u>Unsupervised Learning</u>: Algorithm builds a model on *unlabled* data, no way to see if it did what it was "supposed to do."

**Example**: Let's return to the customer data example we discussed in [Lecture 2](https://colab.research.google.com/drive/1JFhy1B1iBXe2pWviXMeZmN-k6l_erQ5C?usp=sharing). This version of the dataset is a little different:

A  | B  |  C  |  D
----|----|----|----
2 | 3 | 9 | 1
4 | 6 | 14 | 0
3 | 7 | 17 | 0
5 | 2 | 10 | 1
3 | 2 | 7 | 1


1. $C$ isn't *exactly* $1A$ + $2B$ like it was in the original version. If we let $C$ be the "acutal" value and denote the *predicted* $C$ with $\hat{C}$ let We could write $\hat{C} = 1A + 2B$. This is a *Linear Regression Model*. **Discuss**. How can we think of the above definitions in this context?

2. There's a new column for variable $D$. The values appear numeric but they represent a categorical variable. A *Logistic Regression* model performs a "classification" task: $\hat{p}(D)= f(A, B, C)$. **Discuss**. How can we think of the above definitions in this context?  

3. What if we didn't observe $D$ but suspect that there is some kind of fundamental underlying "groupings" among the observations? Is there a way to predict which observations are in the "0" group and which ones are in the "1" group without being able to check our work with variable $D$?

*Linear Regression* and *Logistic Regression* are in a larger class of models called *General Linear Models*, which are covered in chapters 11 and 12 of *Practical Linear Algebra for Data Science*.







## 5.2 The k-Means Clustering Algorithm

**Goal**: Group observations from a dataset into "similar" groups *without supervision*.

#### Model Training.

In supervised learning we define an objective function that uses the labeled outcomes in our dataset to check model performance. In this case, the job of the *objective function* is to train the model by predicting the test data's labels as accurately as possible.

**Discuss**. What is a reasonable objective function for a Linear Regression model? Hint: [Homework 3](https://colab.research.google.com/drive/19jiDN30WNHJhWOJQ3iojso6bC2NZ23eP?usp=sharing)!

Since K-Means is an unsupervised method we need to be creative in coming up with the objective function ourselves. This can be done in many different ways. The most common is using (Eculidian) distance to measure whether or not a particular grouping is close to a "group representative," often called a "centroid."  


#### The k-Means Objective Function.

Let $\vec{x}_i$ be the $i^{th}$ \_\_\_\_\_\_\_\_\_ vector in the dataset.

If $\vec{z}_1, ..., \vec{z}_k$ are the group centroid vectors with $j \in {1, ... , k}$, and $c_i = j$, we define the objective function as

$J^{clust} = (||\vec{x}_1 - \vec{z}_{c_1}||^2 + ||\vec{x}_2 - \vec{z}_{c_2}||^2 + ... + ||\vec{x}_n - \vec{z}_{c_n}||^2)/n.$

**Question**. What does $||\vec{x}_i - \vec{z}_j||$ tell us? This means that we have found an "optimal" grouping when $J^{clust}$ is \_\_\_\_\_\_\_\_\_\_.

We can miminize $J^{clust}$ by minimizing each inidividual $||\vec{x}_i - \vec{z}_{c_i}||$. Thus, the optimal grouping solution is found by

$\begin{align} \underset{j = 1, 2, ..., k}{min} ||\vec{x}_1 - \vec{z}_j||^2 + \underset{j = 1, 2, ..., k}{min}||\vec{x}_2 - \vec{z}_j||^2 + ... + \underset{j = 1, 2, ..., k}{min}||\vec{x}_n - \vec{z}_j||^2.\end{align}$


What is the process for finding the optimal $J^{clust}$?

**k-Means Clustering Algorithm (Lloyd's)**:

1. Choose *k*, the number of groups. Randomly initialize *k* centroids.
2. Compute the Euclidian distance between each data observation (vector) and each of the *k* centroids.
3. Assign each vector to the group with the closest centroid.
4. Compute $J^{clust}$.
5. Update each centroid as the average of all vectors assigned to that centroid's group.
6. Repeat steps 2-5 until convergence.  

**Definitions**.

<u>Iteration</u> the process of repeating a set of instructions or operations until a condition is met. This is typically implemented by looping. An iteration count tells you how many times you've gone through the loop.

An algorithm has <u>converged</u> if repeating the algorithm's steps leads to no (or no significant by a predefined armount) change in the algorithm's objective function.



## 5.3 k-Means From Scratch

The philosophy behind how this course is designed is roughly:

a) work out theoretical mathematical concepts by-hand (on the board or on paper), then

b) implement these ideas using mostly pre-written functions from libraries such as ```numpy``` and ```sklearn``` in Python.

The idea of coding "from scratch" is to write code with minimal reliance on libraries. From a computational perspective this is inneficient, but from a "understanding how to code and how to get Python to implement theoretical mathematics" perspective it's considered a best-practice for learners.

If you're interested in improving your coding skills and underlying knowledge of the concepts we're studying in this course I strongly recommend the text [Data Science From Scratch: First Principles with Python](https://www.amazon.com/Data-Science-Scratch-Principles-Python-dp-1492041130/dp/1492041130/ref=dp_ob_title_bk) by Joel Grus.



**Example**: Let's explore how the k-means algorithm works with a small dataset and doing the algorithm's steps "from scratch."

Here's the data:

A  | B  
----|----
1 | 1
1 | 0
0 | 2
2 | 4
3 | 5



Our goal is to put each observation vectors into one of two groups. Assume that $\vec{z}_0 = \vec{x}_0$ and $\vec{z}_1 = \vec{x}_2$ are randomly chosen to be the initial group centroids (which step is this?). Assume that the variables were measured on the same scale (more on this later).

Here are the individual row vectors we'll be grouping:

$\vec{x}_0 = [1, 1]$,
$\vec{x}_1 = [1, 0]$,
$\vec{x}_2 = [0, 2]$,
$\vec{x}_3 = [2, 4]$,
$\vec{x}_4 = [3, 5]$.


In [ ]:
# Let's work in Python to save us some time as compared to doing this fully by-hand.
import numpy as np
X = np.array([[1, 1], [1,0], [0, 2], [2, 4], [3, 5]])
print(X)

In [ ]:
# Separate and assign vector names to each row vector.
x_0 =
x_1 =
x_2 =
x_3 =
x_4 =

x_3

In [ ]:
# Step 1: "Randomly" initialize the k = 2 centroids. Note that group labels are 0 and 1. This step only needs to be done once (why?).
z_0 =
z_1 =

print(z_0)
print(z_1)

### Iteration 0.

In [ ]:
# Step 2: Which group should we assign each vector to?
print(np.linalg.norm(?? - z_0)**2)
print(np.linalg.norm(?? - z_1)**2)

# Step 3: Assign each vector to a group.
# group 0:
# group 1:

In [ ]:
# Step 4: Compute J_clust.

J_0 = (np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2)/?
print(J_0)

In [ ]:
# Step 5: Update the centroids.

z_0 =
z_1 =
print(z_0)
print(z_1)

### Iteration 1.

In [ ]:
# Step 2: Centroids were updated so re-do the assignments. Did the groups change?

print(np.linalg.norm(?? - z_0)**2)
print(np.linalg.norm(?? - z_1)**2)

# Step 3: Update assignments.
# group 0:
# group 1:

In [ ]:
# Step 4: Compute J_clust. What do you expect will happen here?

J_1 =(np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2)/?

print(J_1)

In [ ]:
# Step 5: Update the centroids.

z_0 =
z_1 =

print(z_0)
print(z_1)

### Iteration 2.

In [ ]:
# Step 2.

print(np.linalg.norm(?? - z_0)**2)
print(np.linalg.norm(?? - z_1)**2)

# Step 3.
# group 0:
# group 1:

In [ ]:
# Step 4: Update J_clust.

J_2 =(np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2)/?

print(J_2)

In [ ]:
# Step 5: "Update" centroids.

z_0 =
z_1 =

print(z_0)
print(z_1)

### Iteration 3.

In [ ]:
# Step 2.

print(np.linalg.norm(?? - z_0)**2)
print(np.linalg.norm(?? - z_1)**2)

# Step 3.
# group 0:
# group 1:

In [ ]:
# Step 4: Update J_clust. Group membership didn't change this time. Will J_3 be less than J_2?

J_2 =(np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2 + np.linalg.norm(??)**2)/?

print(J_2)

In [ ]:
# Step 5: "Update" centroids. # Is this necessary?

z_0 =
z_1 =

print(z_0)
print(z_1)

## 5.4 Conclusions

* We could run the algorithm once more to verify that $J^{clust}$ has converged. What would $J_3$ be and why?
* Why is this alogorithm "unsupervised" and what are we doing to "check" its work? How is this fundamentally related to the centroids "updating" each iteration?
* Why did we need to include the assumption that "the variables were measured on the same scale?" What could we do if this wasn't the case?
* How could we use a for-loop or while-loop to further automate this process?

## 5.5 Algorithms, Convergence, and While-Loops

Let's take a look at implementing a simple algorithm that uses a while-loop to do the computation. This topic is important, but optional, and is closely related to **Exercise 12** in [Homework 5](https://colab.research.google.com/drive/1nm-lpzl7B3cKvyuu2160WXcs7d7L0iJJ?usp=sharing).

In [ ]:
# Here's a while-loop implementing a simple "Guess the Number" algorithm.
# Algorithm has "converged" once the absolute difference between the guess and the actual number is less than or equal to 1.

number = 81 # This initializes the actual number. It's the "ground truth" data used to check the work. Is this "supervised" or "unsupervised" learning?
guess = 71 # This initializes our first guess, made at random. Think of it as guess_0.
difference = guess - number # This initializes the difference variable, which will update each iteration and be used to determine if the algorithm converged.
iteration_count = 0 # This initializes a variable that counts how many times the while-loop has run. Allows us to exit loop if it runs too many times and doesn't converge.

# Note that all of this is for illustration purposes: Even if "guess" was somehow fully hidden once we know what "difference" is we can deduce what "number" is.
# The point of this is to illustrate implementing an algorithm that utilizes a while-loop to do the work.

while(abs(difference) > 1): # Guess is too far from number, not sure which direction.
  if(difference > 1): # Guess was too high.
    guess = guess - 1 # Update guess to be a little lower.
  if(difference < -1): # Guess was too low.
    guess = guess + 1 # Update guess to be a little higher.
  difference = guess - number # Update the difference variable so it can get re-checked by the next run in the while-loop.
  iteration_count = iteration_count + 1 # The loop ran once so update the count.
  if(iteration_count >= 1000):
    break # This stops the loop once we have guessed 1000 times. Could set to whatever value we want.

print(f'Guess is {guess} after {iteration_count} iterations.' ) # This will tell us what the algorithm settled on for its final guess after it either converged or ran out of iterations.
# See page 296 of Practical Linear Algebra for more examples on print formatting and F-Strings, which is being utilized here to make our results more readable.
